# Propensity to Donate: Model Evaluation & Cross-Validation
Following the textbook guidelines for small/medium datasets, we will freeze a final test set once, then use cross-validation on the training data for model selection and tuning. We will compare different cross-validation strategies (`KFold`, `StratifiedKFold`, and `RepeatedStratifiedKFold`) to see which provides the most reliable estimate for our class distribution, and then use the best one in a `GridSearchCV`.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from datetime import timedelta
from supabase import create_client, Client
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    RepeatedStratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# ── Supabase connection ───────────────────────────────────────────────────────
# Set SUPABASE_URL and SUPABASE_KEY as environment variables before running.
SUPABASE_URL = os.environ.get('SUPABASE_URL', '')
SUPABASE_KEY = os.environ.get('SUPABASE_KEY', '')
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

def fetch_table(table: str) -> pd.DataFrame:
    resp = supabase.table(table).select('*').execute()
    return pd.DataFrame(resp.data) if resp.data else pd.DataFrame()

def to_snake(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower() for c in df.columns]
    return df

# ── Load raw tables ───────────────────────────────────────────────────────────
supporters = to_snake(fetch_table('Supporters'))
donations  = to_snake(fetch_table('Donations'))

donations['donation_date'] = pd.to_datetime(donations['donation_date'], errors='coerce')
donations = donations.dropna(subset=['donation_date', 'estimated_value'])
supporters['created_at'] = pd.to_datetime(supporters['created_at'], errors='coerce')

print(f'Supporters: {len(supporters)} | Donations with value: {len(donations)}')

# ── Build dataset using rolling 90-day observation windows ───────────────────
# For each observation date, compute RFM features for every supporter
# who has donated before that date, then label whether they donated
# in the following 90 days. This creates multiple rows per supporter
# and dramatically increases dataset size vs. a single snapshot.

WINDOW_DAYS = 90
min_date = donations['donation_date'].min() + timedelta(days=WINDOW_DAYS)
max_date = donations['donation_date'].max() - timedelta(days=WINDOW_DAYS)
observation_dates = pd.date_range(start=min_date, end=max_date, freq='90D')

rows = []
for obs_date in observation_dates:
    future_end = obs_date + timedelta(days=WINDOW_DAYS)

    prior  = donations[donations['donation_date'] <  obs_date]
    future = donations[
        (donations['donation_date'] >= obs_date) &
        (donations['donation_date'] <  future_end)
    ]['supporter_id'].unique()

    if prior.empty:
        continue

    rfm = prior.groupby('supporter_id').agg(
        recency        = ('donation_date',  lambda x: (obs_date - x.max()).days),
        frequency      = ('donation_id',    'count'),
        monetary_total = ('estimated_value', 'sum'),
        monetary_avg   = ('estimated_value', 'mean'),
        is_recurring   = ('is_recurring',    lambda x: int(x.any())),
    ).reset_index()

    rfm['target_donated_next_90_days'] = rfm['supporter_id'].isin(future).astype(int)
    rfm['obs_date'] = obs_date
    rows.append(rfm)

dataset = pd.concat(rows, ignore_index=True)

# ── Merge static supporter features ──────────────────────────────────────────
sup_static = supporters[[
    'supporter_id', 'supporter_type', 'relationship_type',
    'acquisition_channel', 'region'
]].copy()

dataset = dataset.merge(sup_static, on='supporter_id', how='left')
dataset = dataset.drop(columns=['supporter_id', 'obs_date'])

# ── X / y split ──────────────────────────────────────────────────────────────
y = dataset['target_donated_next_90_days']
X = dataset.drop(columns=['target_donated_next_90_days'])

print(f'\nDataset rows:    {len(dataset)}')
print(f'Positive rate:   {y.mean():.1%}  (donated in next 90 days)')
print(f'Features ({len(X.columns)}): {list(X.columns)}')
dataset.head()


Supporters: 60 | Donations with value: 420

Dataset rows:    549
Positive rate:   41.5%  (donated in next 90 days)
Features (9): ['recency', 'frequency', 'monetary_total', 'monetary_avg', 'is_recurring', 'supporter_type', 'relationship_type', 'acquisition_channel', 'region']


,recency,frequency,monetary_total,monetary_avg,is_recurring,target_donated_next_90_days,supporter_type,relationship_type,acquisition_channel,region
0,15,1,774.61,774.61,1,1,SocialMediaAdvocate,Local,SocialMedia,Luzon
1,32,1,2565.03,2565.03,0,0,Volunteer,Local,SocialMedia,Mindanao
2,46,1,250.00,250.00,1,1,MonetaryDonor,Local,SocialMedia,Luzon
3,25,1,439.51,439.51,1,0,MonetaryDonor,PartnerOrganization,Church,Mindanao
4,18,1,11.87,11.87,0,1,MonetaryDonor,International,WordOfMouth,Visayas


## 1. Train / Test Split (Hold-out Set)
We split the data 80/20. We use `stratify=y` to ensure our test set has the same proportion of churned/retained donors as our training set.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (439, 9)
Test set size: (110, 9)


## 2. Preprocessing Pipeline
As recommended in the textbook, we process numeric and categorical variables inside a pipeline to prevent data leakage during cross-validation.

In [6]:
numeric_features = X.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create a baseline Random Forest model pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

## 3. Compare Cross-Validation Methods
We test three methods from the textbook:
- **KFold:** Standard random splits.
- **StratifiedKFold:** Preserves the percentage of samples for each class.
- **RepeatedStratifiedKFold:** Repeats Stratified K-Fold multiple times for a more stable estimate on small datasets.

In [7]:
cv_methods = {
    'KFold (5 splits)': KFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold (5 splits)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedStratifiedKFold (5 splits, 3 repeats)': RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
}

print("Cross-Validation ROC AUC Scores on Training Data:")
for name, cv in cv_methods.items():
    # We use roc_auc as our scoring metric to account for class balance
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

Cross-Validation ROC AUC Scores on Training Data:


KFold (5 splits): 0.5600 +/- 0.0206
StratifiedKFold (5 splits): 0.5832 +/- 0.0345
RepeatedStratifiedKFold (5 splits, 3 repeats): 0.5788 +/- 0.0474


## 4. Hyperparameter Tuning with the Best CV Method
Because our dataset is small, `RepeatedStratifiedKFold` generally gives the most stable evaluation. We will use it inside `GridSearchCV` to find the best hyperparameters for our Random Forest.

In [8]:
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 3, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 5]
}

best_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=best_cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validated ROC AUC: {grid_search.best_score_:.4f}")

Best Hyperparameters: {'classifier__max_depth': 3, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 100}
Best Cross-Validated ROC AUC: 0.6113


## 5. Final Evaluation on the Frozen Test Set
Now that we have selected our best model using cross-validation on the training set, we evaluate it **once** on the held-out test set to get an unbiased estimate of its generalization performance.

In [9]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Final Test Set Evaluation:\n")
print(classification_report(y_test, y_pred))
print(f"Test ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

Final Test Set Evaluation:

              precision    recall  f1-score   support

           0       0.67      0.84      0.74        64
           1       0.66      0.41      0.51        46

    accuracy                           0.66       110
   macro avg       0.66      0.63      0.63       110
weighted avg       0.66      0.66      0.65       110

Test ROC AUC: 0.6405


In [10]:
import joblib

# Export the best model pipeline 
# (This saves the Random Forest AND the preprocessing steps!)
filename = "propensity_to_donate_pipeline.pkl"
joblib.dump(best_model, filename)

print(f"✅ Successfully exported the entire prediction pipeline!")
print(f"📦 Saved as: {filename}")

✅ Successfully exported the entire prediction pipeline!
📦 Saved as: propensity_to_donate_pipeline.pkl
